# Data Analysis and Knowledge Management — European Urban Centres Dataset

**Author:** Filipa Neves  
**Institution:** Instituto Superior de Engenharia do Porto (ISEP)  
**Course:** Data Analysis and Knowledge Management (ANDOC)  

---

## Project Overview

This notebook presents a complete data science pipeline applied to a dataset of **1,165 European urban centres**, containing 33 variables covering economic, social, demographic and infrastructural indicators.

The project is structured in three main stages:

1. **Data Understanding** — exploratory analysis, outlier detection and data quality assessment  
2. **Data Preparation** — missing value imputation, encoding, log transformation and normalisation  
3. **Predictive Modelling** — two supervised learning tasks:
   - **Regression:** predicting average GDP (`SC_GDP_AVG_2020`)
   - **Classification:** predicting Human Development Index class (`SC_SEC_HDI_2020`)

Models used: K-Nearest Neighbours (KNN), Decision Tree, Random Forest

---


## 1. Data Understanding

### 1.1 Loading the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set(style="whitegrid")

# Load dataset — update the file path as needed
file_path = "europeCitiesData.xlsx"
df = pd.read_excel(file_path, sheet_name="data")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

### 1.2 Dataset Overview

In [ ]:
# Display column types and non-null counts
df.info()


**Dataset summary:**
- Total records: 1,165 urban centres
- Total variables: 33
- Numerical variables: 30
- Categorical variables: 3 (urban centre name, country, economic group)
- Some missing values detected in certain columns


In [ ]:
# First 5 rows
print("First 5 rows:")
display(df.head())

In [ ]:
# Descriptive statistics for numerical variables
print("Descriptive statistics:")
display(df.describe().T)

In [ ]:
# Identify numerical and categorical variables
num_vars = df.select_dtypes(include=[np.number]).columns
cat_vars = df.select_dtypes(exclude=[np.number]).columns

print(f"Number of numerical variables: {len(num_vars)}")
print(f"Number of categorical variables: {len(cat_vars)}")
print("Categorical variables:", list(cat_vars))

### 1.3 Outlier Analysis

In [ ]:
# Outlier detection using IQR and Z-score methods for all numerical variables
num_vars = df.select_dtypes(include=[np.number]).columns
outlier_summary = []

for var in num_vars:
    Q1 = df[var].quantile(0.25)
    Q3 = df[var].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    mean = df[var].mean()
    std = df[var].std()
    z_scores = (df[var] - mean) / std

    # Identify outliers by each method
    outliers_iqr = df[(df[var] < lower_bound) | (df[var] > upper_bound)][var]
    outliers_z = df[np.abs(z_scores) > 3][var]

    outlier_summary.append({
        "Variable": var,
        "Mean": round(mean, 3),
        "Std Dev": round(std, 3),
        "Lower Bound (IQR)": round(lower_bound, 3),
        "Upper Bound (IQR)": round(upper_bound, 3),
        "Outliers (IQR)": len(outliers_iqr),
        "Outliers (Z-score)": len(outliers_z),
        "Max": round(df[var].max(), 3),
        "Min": round(df[var].min(), 3)
    })

outlier_df = pd.DataFrame(outlier_summary)
display(outlier_df)

In [ ]:
# Consistency check for HL_ variables
# Proportion variables (HL_FPC_) must be between 0 and 1
# Count/density variables must be non-negative

hl_cols = [col for col in df.columns if col.startswith("HL_")]
prop_cols = [col for col in hl_cols if col.startswith("HL_FPC_")]
count_density_cols = [col for col in hl_cols if col not in prop_cols]

errors = []

# Check proportion variables
for col in prop_cols:
    invalid = df[(df[col] < 0) | (df[col] > 1)][col]
    if not invalid.empty:
        errors.append({
            "Variable": col,
            "Type": "Proportion [0,1]",
            "Invalid Count": len(invalid),
            "Min Found": invalid.min(),
            "Max Found": invalid.max()
        })

# Check count/density variables for negative values
for col in count_density_cols:
    negatives = df[df[col] < 0][col]
    if not negatives.empty:
        errors.append({
            "Variable": col,
            "Type": "Count/Density (>=0)",
            "Invalid Count": len(negatives),
            "Min Found": negatives.min(),
            "Max Found": negatives.max()
        })

if errors:
    print("HL_ variables with inconsistent values:")
    display(pd.DataFrame(errors))
else:
    print("All HL_ variables are consistent with their expected value ranges.")

### 1.4 Data Type Consistency Check

In [ ]:
# Check if any object-type columns contain numeric values (possible encoding errors)
issues = []

for col in df.columns:
    if df[col].dtype == "object":
        temp = pd.to_numeric(df[col], errors="coerce")
        n_numeric = temp.notna().sum()
        if n_numeric > 0 and n_numeric / len(df) > 0.3:
            issues.append(col)

if issues:
    print("Columns with possible data type inconsistencies:")
    for col in issues:
        print(f" - {col}")
else:
    print("All column types are consistent with their content.")


### 1.5 Key Observations

**Dataset characteristics:**
- 1,165 observations and 33 variables representing European urban centres
- 30 numerical variables and 3 categorical variables:
  - `GC_UCN_MAI_2020`: urban centre name
  - `GC_CNT_GAD_2020`: country
  - `GC_DEV_WIG_2020`: economic group (e.g. High income, Upper middle income)

**Target variables:**
- `SC_GDP_AVG_2020`: continuous variable used for regression
- `SC_SEC_HDI_2020`: continuous variable (0.64–0.98) used for 4-class classification

**Key findings:**
- Average GDP: mean ~72M USD, median ~60M USD — strong positive skew (major capitals as outliers)
- HDI: mean ~0.871, std ~0.057 — relatively concentrated distribution
- Population: mean ~259K, max ~14.3M — high variability between small and large cities
- Variables `SC_SEC_HDI_2020`, `SC_SEC_LET_2020`, `SC_SEC_SYT_2020` have only 2 missing values
- Pharmaceutical facility variables (PHA) have ~40% missing values — require imputation or exclusion

**Pre-processing requirements identified:**
- Log transformation recommended for highly skewed variables (GDP, population, area)
- Standardisation required before distance-based models (KNN)
- Normalisation needed due to variables at very different scales


### 1.6 Exploratory Visualisations

In [ ]:
# GDP distribution
plt.figure(figsize=(7, 5))
sns.histplot(df["SC_GDP_AVG_2020"], bins=30, kde=True, color='#7B1FA2')
plt.title("Distribution of Average GDP (2020)")
plt.xlabel("Average GDP (USD)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# GDP by economic group
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="GC_DEV_WIG_2020", y="SC_GDP_AVG_2020", color='#F4B6C2')
plt.title("Average GDP by Economic Group (2020)")
plt.xlabel("Economic Group")
plt.ylabel("Average GDP (USD)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# HDI distribution
plt.figure(figsize=(7, 5))
sns.histplot(df["SC_SEC_HDI_2020"], bins=20, kde=True, color='#D81B60')
plt.title("Distribution of HDI (2020)")
plt.xlabel("HDI")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: GDP vs HDI by economic group
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=df,
    x="SC_GDP_AVG_2020",
    y="SC_SEC_HDI_2020",
    hue="GC_DEV_WIG_2020",
    palette=["#F4B6C2", "#D81B60", "#7B1FA2"],
    s=30,
    alpha=0.85
)
plt.title("GDP vs HDI by Economic Group (2020)", fontsize=12)
plt.xlabel("Average GDP (USD)", fontsize=10)
plt.ylabel("HDI", fontsize=10)
plt.legend(title="Economic Group", fontsize=8, title_fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# GDP evolution over time (1990-2020)
gdp_cols = ["SC_GDP_AVG_1990", "SC_GDP_AVG_1995", "SC_GDP_AVG_2000",
            "SC_GDP_AVG_2005", "SC_GDP_AVG_2010", "SC_GDP_AVG_2015", "SC_GDP_AVG_2020"]

gdp_long = df.melt(value_vars=gdp_cols, var_name="Year", value_name="Average_GDP")
gdp_long["Year"] = gdp_long["Year"].str.extract(r"(\d{4})").astype(int)
df_yearly_mean = gdp_long.groupby("Year")["Average_GDP"].mean().reset_index()

plt.figure(figsize=(7, 5))
sns.lineplot(data=df_yearly_mean, x="Year", y="Average_GDP", marker="o", color="#FFB6A0", linewidth=2.5)
plt.title("Average GDP Evolution in European Cities (1990-2020)")
plt.xlabel("Year")
plt.ylabel("Average GDP (USD)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Area vs Population and Population Density
df["Population_Density"] = df["GC_POP_TOT_2020"] / df["GC_UCA_KM2_2020"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.scatterplot(data=df, x="GC_UCA_KM2_2020", y="GC_POP_TOT_2020",
                color="#F4B6C2", alpha=0.8, ax=axes[0])
axes[0].set_title("Area vs Population (2020)")
axes[0].set_xlabel("Area (km²)")
axes[0].set_ylabel("Total Population")
axes[0].grid(True, linestyle="--", alpha=0.5)

sns.histplot(df["Population_Density"], bins=30, kde=True, color="#FFB6A0", ax=axes[1])
axes[1].set_title("Population Density Distribution (2020)")
axes[1].set_xlabel("Inhabitants per km²")
axes[1].set_ylabel("Frequency")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix and identification of highly correlated variables
corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

# Identify variables with correlation > 0.95
corr_abs = corr_matrix.abs()
upper_tri = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.95)]

print("Variables to remove due to high correlation (> 0.95):")
print(to_drop)

## 2. Data Preparation

### 2.1 Missing Value Imputation

In [ ]:
# Simple imputation with median for variables with very few missing values
simple_impute_cols = ['SC_SEC_HDI_2020', 'SC_SEC_LET_2020', 'SC_SEC_SYT_2020']

for col in simple_impute_cols:
    df[col] = df[col].fillna(df[col].median())

print("Median imputation applied to:", simple_impute_cols)

# Conditional imputation by country for pharmaceutical facility variables (~40% missing)
pha_cols = ['HL_FCL_PHA_2020', 'HL_FDE_PHA_2020', 'HL_FPC_PHA_2020']

for col in pha_cols:
    df[col] = df.groupby('GC_CNT_GAD_2020')[col].transform(lambda x: x.fillna(x.median()))

# Fallback: global median for countries with no data
df[pha_cols] = df[pha_cols].fillna(df[pha_cols].median())

print("Conditional imputation (by country) applied to:", pha_cols)

# Final check
print("
Remaining missing values after imputation:")
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
# Conditional imputation for hospital facility variables (HOS)
hos_cols = ['HL_FCL_HOS_2020', 'HL_FDE_HOS_2020', 'HL_FPC_HOS_2020']

for col in hos_cols:
    df[col] = df.groupby('GC_CNT_GAD_2020')[col].transform(lambda x: x.fillna(x.median()))

df[hos_cols] = df[hos_cols].fillna(df[hos_cols].median())

print("Missing values after final imputation:")
print(df.isna().sum()[df.isna().sum() > 0])

### 2.2 Categorical Variable Encoding

In [ ]:
# Check unique values in economic group variable
print(df['GC_DEV_WIG_2020'].unique())

In [ ]:
# Ordinal encoding for economic group variable
# Lower Middle = 1, Upper Middle = 2, High income = 3
df['GC_DEV_WIG_2020'] = df['GC_DEV_WIG_2020'].map({
    'Lower Middle': 1,
    'Upper Middle': 2,
    'High income': 3
})

print("Ordinal encoding applied to GC_DEV_WIG_2020:")
print(df['GC_DEV_WIG_2020'].value_counts(dropna=False))

### 2.3 Log Transformation

In [ ]:
# Apply log1p transformation to highly skewed variables (avoids log(0))
cols_to_log = ['GC_POP_TOT_2020', 'GC_UCA_KM2_2020', 'SC_GDP_AVG_1990',
               'SC_GDP_AVG_1995', 'SC_GDP_AVG_2020']

for col in cols_to_log:
    df[f'{col}_log'] = np.log1p(df[col])

print("Log transformation applied to:", cols_to_log)

# Visualise distributions before and after transformation
for col in cols_to_log:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.histplot(df[col], bins=30, kde=True, color="#F4B6C2", ax=axes[0])
    sns.histplot(df[f'{col}_log'], bins=30, kde=True, color="#FFB6A0", ax=axes[1])
    axes[0].set_title(f"Original: {col}")
    axes[1].set_title(f"After log transform: {col}_log")
    plt.tight_layout()
    plt.show()

In [ ]:
# Save a copy of the dataset before normalisation (used for GDP regression)
df_gdp = df.copy()
df.head()

### 2.4 Feature Standardisation

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns
num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

# Exclude ordinal, categorical and target variables from standardisation
cols_to_exclude = [
    'GC_DEV_WIG_2020',   # Ordinal variable
    'SC_GDP_AVG_2020',   # Regression target
    'SC_SEC_HDI_2020'    # Classification target
]

cols_to_scale = [col for col in num_cols if col not in cols_to_exclude]

# Apply Z-score standardisation
scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

print(f"Standardisation applied to {len(cols_to_scale)} numerical variables.")
print("Excluded variables:", cols_to_exclude)

In [ ]:
df.head()

In [ ]:
from scipy.stats import shapiro

# Normality test on log-transformed variables
for col in ['GC_POP_TOT_2020_log', 'SC_GDP_AVG_2020_log']:
    stat, p = shapiro(df[col])
    print(f"{col}: p-value = {p:.6f}")

### 2.5 Removal of Highly Correlated Variables

In [ ]:
# Remove redundant variables (correlation > 0.95), keeping SC_GDP_AVG_2020 as target
vars_to_remove = ['SC_GDP_AVG_2000', 'SC_GDP_AVG_2005', 'SC_GDP_AVG_2010',
                  'SC_GDP_AVG_2015', 'IN_ROA_LEN_2020']

df = df.drop(columns=vars_to_remove)

print("Variables removed due to high correlation:")
print(vars_to_remove)

## 3. Predictive Modelling

### 3.1 Classification — HDI Prediction

#### 3.1.1 Target Creation and Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Create 4 HDI classes based on quartiles (0=Low, 1=Medium, 2=High, 3=Very High)
col_hdi_name = [c for c in df.columns if 'SC_SEC_HDI' in c][0]
df['HDI_Class'] = pd.qcut(df[col_hdi_name], q=4, labels=[0, 1, 2, 3])

# Remove all HDI-related columns from features to prevent data leakage
cols_to_drop_hdi = ['GC_UCN_MAI_2020', 'HDI_Class']
for col in df.columns:
    if 'HDI' in col and col not in cols_to_drop_hdi:
        cols_to_drop_hdi.append(col)

# Prepare clean feature set
X_final = df.drop(columns=cols_to_drop_hdi, errors='ignore')

# Remove remaining non-numeric columns
cols_text = X_final.select_dtypes(exclude=['number']).columns.tolist()
X_final = X_final.drop(columns=cols_text, errors='ignore')

# Apply one-hot encoding if needed
categorical_cols = ['GC_CNT_GAD_2020', 'GC_DEV_WIG_2020']
valid_cat_cols = [c for c in categorical_cols if c in X_final.columns]
X_final = pd.get_dummies(X_final, columns=valid_cat_cols, dtype=int)

y_final = df['HDI_Class']

# Stratified split to preserve class balance
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_final, y_final, test_size=0.3, random_state=42, stratify=y_final
)

print("Data ready for HDI classification.")
print(f"Training set shape: {X_train_h.shape}")
print(f"Test set shape: {X_test_h.shape}")
print(X_train_h.dtypes.value_counts())

#### 3.1.2 K-Nearest Neighbours Classifier

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Test k values from 1 to 20
k_values = range(1, 21)
accuracy_list = []

for k in k_values:
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    pipeline.fit(X_train_h, y_train_h)
    y_pred = pipeline.predict(X_test_h)
    acc = accuracy_score(y_test_h, y_pred)
    accuracy_list.append(acc)
    print(f"k={k}: Accuracy={acc:.4f}")

# Select best k
best_k = k_values[np.argmax(accuracy_list)]
print(f"
Best k based on accuracy: {best_k}")

# Train final model
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=best_k))
])
final_pipeline.fit(X_train_h, y_train_h)
y_final_pred = final_pipeline.predict(X_test_h)

final_acc = accuracy_score(y_test_h, y_final_pred)
print(f"
Final Results (k={best_k}):")
print(f"Accuracy: {final_acc:.2%}")
print("-" * 30)
print(classification_report(y_test_h, y_final_pred))

# Accuracy vs k
plt.figure(figsize=(12, 6))
plt.plot(k_values, accuracy_list, marker='o', label='Accuracy', color='green')
plt.xlabel('k (Number of Neighbours)')
plt.ylabel('Accuracy')
plt.title('KNN Classification: Accuracy vs k')
plt.legend()
plt.grid(True)
plt.show()

# Confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test_h, y_final_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'Medium', 'High', 'Very High'],
            yticklabels=['Low', 'Medium', 'High', 'Very High'])
plt.title(f'Confusion Matrix — KNN (k={best_k})')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

#### 3.1.3 Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

# Prepare dataset (ensure y is integer type)
cols_to_drop_tree = [c for c in df.columns if 'HDI' in c and c != 'HDI_Class'] +                     df.select_dtypes(include=['object']).columns.tolist()

X_safe = df.drop(columns=cols_to_drop_tree + ['HDI_Class'], errors='ignore').fillna(0)
y_safe = df['HDI_Class'].astype(int)

X_train_tree, X_test_tree, y_train_tree, y_test_tree = train_test_split(
    X_safe, y_safe, test_size=0.3, random_state=42, stratify=y_safe
)

# Evaluate different tree depths
def evaluate_decision_tree_hdi(X_train, y_train, X_test, y_test, depth_values):
    results = []
    for depth in depth_values:
        model = DecisionTreeClassifier(max_depth=depth, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results.append({'max_depth': depth, 'Accuracy': acc})
    return pd.DataFrame(results)

depth_values = range(1, 16)
results_df = evaluate_decision_tree_hdi(X_train_tree, y_train_tree, X_test_tree, y_test_tree, depth_values)

print("Results by depth:")
print(results_df)

best_index = results_df['Accuracy'].idxmax()
best_depth = results_df.loc[best_index, 'max_depth']
best_acc = results_df.loc[best_index, 'Accuracy']
print(f"
Best depth: {best_depth} (Accuracy: {best_acc:.2%})")

# Train best model
best_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
best_tree.fit(X_train_tree, y_train_tree)

# Visualise tree (up to depth 3 for readability)
plt.figure(figsize=(20, 10))
plot_tree(best_tree,
          feature_names=X_safe.columns,
          class_names=['Low', 'Medium', 'High', 'Very High'],
          filled=True, rounded=True, fontsize=10, max_depth=3)
plt.title(f"Decision Tree — HDI Classification (Depth: {best_depth})")
plt.show()

# Feature importance
importances = best_tree.feature_importances_
indices = np.argsort(importances)[::-1]
top_n = 10

plt.figure(figsize=(12, 6))
plt.bar(range(top_n), importances[indices[:top_n]], align="center", color='teal')
plt.xticks(range(top_n), X_safe.columns[indices[:top_n]], rotation=45, ha='right')
plt.title("Top 10 Most Important Features for HDI Classification")
plt.tight_layout()
plt.show()

#### 3.1.4 Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Remove identifier column if present
if 'ID_UC_G0' in X_train_tree.columns:
    X_train_tree = X_train_tree.drop(columns=['ID_UC_G0'])
    X_test_tree = X_test_tree.drop(columns=['ID_UC_G0'])

# Train Random Forest with 1000 estimators
rf_clf = RandomForestClassifier(n_estimators=1000, random_state=42, n_jobs=-1)
rf_clf.fit(X_train_tree, y_train_tree)

y_pred_rf = rf_clf.predict(X_test_tree)
acc = accuracy_score(y_test_tree, y_pred_rf)

print("-" * 30)
print(f"Random Forest Accuracy: {acc:.2%}")
print("-" * 30)
print(classification_report(y_test_tree, y_pred_rf))

# Confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test_tree, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Low', 'Medium', 'High', 'Very High'],
            yticklabels=['Low', 'Medium', 'High', 'Very High'])
plt.title('Random Forest — HDI Classification')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# Feature importance for Random Forest
importances = rf_clf.feature_importances_
feature_names = X_train_tree.columns

feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False)

print("Top 10 most important features:")
print(feature_imp_df.head(10))

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feature_imp_df.head(15), palette='Greens')
plt.title('Top 15 Most Important Features — Random Forest')
plt.tight_layout()
plt.show()

### 3.2 Regression — GDP Prediction

#### 3.2.1 Dataset Preparation for Regression

In [ ]:
from sklearn.model_selection import train_test_split

# Apply one-hot encoding to country variable
categorical_cols = ['GC_CNT_GAD_2020']
valid_cat_cols = [c for c in categorical_cols if c in df_gdp.columns]
df_gdp = pd.get_dummies(df_gdp, columns=valid_cat_cols, dtype=int)

# Define features and target
y_gdp = df_gdp['SC_GDP_AVG_2020']
X_gdp = df_gdp.drop(columns=['SC_GDP_AVG_2020', 'GC_UCN_MAI_2020', 'SC_GDP_AVG_2020_log'],
                    errors='ignore')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_gdp, y_gdp, test_size=0.3, random_state=42)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

#### 3.2.2 KNN Regressor

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import numpy as np

# Apply standardisation only to continuous variables
continuous_cols = [col for col in X_train.select_dtypes(include=['float64', 'int64']).columns
                   if len(X_train[col].unique()) / len(X_train) > 0.05]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

# Evaluate k values from 1 to 20
k_values = range(1, 21)
mse_scores, mae_scores, r2_scores = [], [], []

for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    mse_scores.append(mean_squared_error(y_test, y_pred))
    mae_scores.append(mean_absolute_error(y_test, y_pred))
    r2_scores.append(r2_score(y_test, y_pred))

results_df = pd.DataFrame({'k': k_values, 'MSE': mse_scores, 'MAE': mae_scores, 'R2': r2_scores})
print(results_df)

best_k = k_values[np.argmin(mse_scores)]
best_r2 = r2_scores[np.argmin(mse_scores)]
best_rmse = np.sqrt(min(mse_scores))

print(f"
Best k: {best_k}")
print(f"R2: {best_r2:.4f}")
print(f"RMSE: {best_rmse:.4f}")

# Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(k_values, mse_scores, marker='o', label='MSE')
axes[0].plot(k_values, mae_scores, marker='s', linestyle='--', label='MAE')
axes[0].set_title('KNN Regression: MSE & MAE vs k')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Error')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(k_values, r2_scores, marker='o', color='orange', label='R2')
axes[1].set_title('KNN Regression: R2 vs k')
axes[1].set_xlabel('k')
axes[1].set_ylabel('R2')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

#### 3.2.3 Decision Tree Regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_decision_tree_regression(X_train, y_train, X_test, y_test, depth_values):
    results = []
    for depth in depth_values:
        model = DecisionTreeRegressor(max_depth=depth, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        results.append({
            'max_depth': depth,
            'MSE': mse,
            'RMSE': np.sqrt(mse),
            'MAE': mean_absolute_error(y_test, y_pred),
            'R2': r2_score(y_test, y_pred)
        })
    return pd.DataFrame(results)

depth_values = range(1, 31)
results_df = evaluate_decision_tree_regression(X_train, y_train, X_test, y_test, depth_values)

print("Metrics by depth:")
print(results_df)

best_depth = results_df.loc[results_df['MSE'].idxmin(), 'max_depth']
print(f"
Best depth (lowest MSE): {best_depth}")

# Train best model
best_tree = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
best_tree.fit(X_train, y_train)

# Visualise tree
plt.figure(figsize=(20, 10))
plot_tree(best_tree, feature_names=X_train.columns,
          filled=True, rounded=True, fontsize=10, max_depth=3)
plt.title(f"Decision Tree — GDP Regression (Max depth displayed: 3 of {best_depth})")
plt.show()

# Feature importance
importances = best_tree.feature_importances_
indices = np.argsort(importances)[::-1]
top_n = 10

plt.figure(figsize=(10, 6))
plt.title("Top 10 Most Important Features for GDP Prediction")
plt.bar(range(top_n), importances[indices[:top_n]], align="center")
plt.xticks(range(top_n), X_train.columns[indices[:top_n]], rotation=45, ha='right')
plt.tight_layout()
plt.show()

#### 3.2.4 Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# Hyperparameter search
param_grid = {'n_estimators': [100], 'max_depth': [None]}
rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print("Best hyperparameters:", grid_search.best_params_)

y_pred = best_rf.predict(X_test)
mse = mean_squared_error(y_test, y_pred)

print("
Random Forest Regressor — Test Set Metrics:")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {np.sqrt(mse):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.4f}")
print(f"R2:   {r2_score(y_test, y_pred):.4f}")

# Feature importance
importances = best_rf.feature_importances_
indices = np.argsort(importances)[::-1]
top_n = 10

plt.figure(figsize=(10, 6))
plt.title("Top 10 Most Important Features — Random Forest Regressor")
plt.bar(range(top_n), importances[indices[:top_n]], align="center")
plt.xticks(range(top_n), X_train.columns[indices[:top_n]], rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("
Top 10 feature importances:")
for i in range(top_n):
    print(f"  {X_train.columns[indices[i]]}: {importances[indices[i]]:.4f}")


## 4. Conclusions

This project applied a complete machine learning pipeline to a dataset of 1,165 European urban centres.

**Data Understanding** revealed significant variability across cities, with GDP and population following a strong positive skew. HDI was more concentrated, reflecting the generally high development levels of European cities. Missing values were limited and manageable.

**Data Preparation** involved median imputation, ordinal encoding, log transformation of skewed variables, Z-score standardisation, and removal of highly correlated features.

**Modelling results:**

For **HDI classification** (4 classes):
- KNN, Decision Tree and Random Forest were compared based on accuracy
- Random Forest achieved the best generalisation performance
- Key predictive features included GDP-related variables and infrastructure indicators

For **GDP regression**:
- KNN, Decision Tree and Random Forest Regressors were evaluated using MSE, RMSE, MAE and R2
- Random Forest provided the best R2 score with lowest error metrics

**Key takeaway:** Ensemble methods (Random Forest) consistently outperformed single models, as expected given the dataset's size and feature diversity.
